<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 1b — Modèles de référence : k-mer + ML classique, et one-hot + CNN
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Représenter une séquence d'ADN par des fréquences de k-mers (k=4) et entraîner une régression logistique<br>
    - Encoder les nucléotides en one-hot et entraîner un petit CNN 1D qui apprend ses propres motifs<br>
    - Comparer les deux références sur accuracy, F1, nombre de paramètres et latence<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Deux façons de transformer une chaîne d'ADN en quelque chose qu'un modèle peut utiliser,
avant même de toucher à un modèle de langage pré-entraîné :

1. **Fréquences de k-mers** : compter la fréquence de chaque sous-chaîne de longueur k
   (par ex. les 256 4-mers possibles), normaliser -> un vecteur de taille fixe -> ML
   classique (régression logistique).
2. **One-hot + CNN** : encoder chaque nucléotide en un vecteur one-hot de dimension 4
   (A/C/G/T), empiler en un tenseur (4, 200), et laisser un petit CNN 1D apprendre
   lui-même ses propres caractéristiques directement à partir de la séquence brute.

<img src="./assets/illustration2.png"/>

In [ ]:
import sys
sys.path.append("src")

import numpy as np
import torch
from data import load_all # importer depuis le dossier src/data.py

splits = load_all("../2-data/processed")
train, val = splits["train"], splits["val"]

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Modèle de référence 1 : fréquences de k-mers + régression logistique**

**À faire :** 
- complétez le code ci-dessous pour calculer les matrices de k-mers (k=4) sur
train/val, 
- entraîner un classifieur logistique (`make_kmer_classifier("logreg")`), 
- puis l'évaluer sur validation avec `evaluate_sklearn`.

In [ ]:
from featurize import kmer_matrix  # importer depuis le dossier src/featurize.py

K = 4
# TODO : calculez les matrices de k-mer pour train et val avec kmer_matrix(..., k=K)
X_train_kmer = ...
X_val_kmer = ...
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

In [ ]:
from models.baselines import make_kmer_classifier  # importer depuis le dossier src/models

# TODO : créez un classifieur avec make_kmer_classifier("logreg") et entraînez-le (fit)
kmer_clf = ...

In [ ]:
from eval import evaluate_sklearn # importer depuis le dossier src/eval.py

# TODO : évaluez-le sur la donnée de validation avec evaluate_sklearn(...)
kmer_metrics = ...
print("k-mer + logreg:", kmer_metrics)

#### **Modèle de référence 2 : nucléotides one-hot + petit CNN**

**À faire :** encodez les fenêtres en one-hot (`one_hot_batch`, longueur 200), instanciez
`OneHotCNN`, puis entraînez-le pendant 5 époques (Adam, lr=1e-3,
`binary_cross_entropy_with_logits`, mini-lots de 256).

In [ ]:
from featurize import one_hot_batch  # importer depuis le dossier src/featurize.py
from models.baselines import OneHotCNN        # importer depuis le dossier src/models
from eval import evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch # importer depuis le dossier src/eval.py

In [ ]:
WINDOW = 200
# TODO : encodez les fenêtres d'entraînement et de validation en one-hot, convertissez en tensors torch
X_train_oh = ...
X_val_oh = ...
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# TODO : instanciez le modèle OneHotCNN(seq_len=WINDOW) et un optimiseur Adam (lr=1e-3)
cnn = ...
optimizer = ...

n = X_train_oh.shape[0]
for epoch in range(5):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        # TODO : calculez les logits du CNN sur ce mini-lot, puis la perte BCE-with-logits
        logits = ...
        loss = ...
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
# TODO : passez X_val_oh dans le CNN (sans gradient) puis évaluez avec evaluate_logits
with torch.no_grad():
    val_logits = ...
cnn_metrics = ...
print("one-hot CNN:", cnn_metrics)

#### **Comparaison**

In [ ]:
import pandas as pd

# TODO : complétez le nombre de paramètres et la latence pour chaque modèle
# (count_params / measure_latency_sklearn pour le k-mer, count_params / measure_latency_torch pour le CNN)
comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": ...,
     "latency_ms": ...},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": ...,
     "latency_ms": ...},
])
comparison

#### **Point de contrôle**

Vous devriez avoir un tableau accuracy/F1/params/latence pour les deux modèles de
référence. Gardez ces chiffres (ou le DataFrame `comparison`) — ils serviront pour le
graphique d'efficacité du Jour 4.

Suite : `02_evo2_embeddings_and_classifier.ipynb` — un modèle de fondation génomique
fait-il mieux ?

*Bloqué ? La version complète est dans `solution/01_kmer_and_cnn_baselines.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 1b</div>
      <div>Prochaine &eacute;tape &rarr; <code>day2/02_evo2_embeddings_and_classifier.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>